#Датирование исторических текстов на нидерландском языке (XIV-XIX вв.)

*Часть работы выполнялась в Google Colab (работа с большими объемами данных), часть - в Visual Studio (вспомогательные этапы работы, связанные с загрузкой/записью файлов в локальном хранилище).*

##1. Загрузка библиотек

In [ ]:
#для работы с датафреймами
import pandas as pd

#для сбора корпуса текстов из директории
import os

#для регулярных выражений
import re

#для работы с массивами
import numpy as np

In [ ]:
#для парсинга .xml-файлов
from bs4 import BeautifulSoup

In [ ]:
#для разбивки текста на батчи при подсчете ttr
from itertools import batched

In [ ]:
#для визуализации данных
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#train_test_split - для разделения данных на train и test и GridSearchCV для подбора параметров модели
from sklearn.model_selection import train_test_split, GridSearchCV, GroupShuffleSplit, StratifiedGroupKFold

#модели
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

#метрики
from sklearn.metrics import accuracy_score, f1_score, classification_report, mean_squared_error, root_mean_squared_error, r2_score, RocCurveDisplay, roc_curve, auc

#для обеспечения баланса классов при использовании MultinomialNB
from sklearn.utils.class_weight import compute_sample_weight

#энкодер для кодирования таргета и скейлер для масштабирования количественных признаков
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

#для создания ансамблей
from sklearn.ensemble import StackingClassifier, BaggingClassifier, GradientBoostingClassifier, AdaBoostClassifier

In [ ]:
# from sklearn.model_selection import GroupKFold

In [ ]:
# from sklearn.model_selection import cross_validate

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
#для подсчета корреляционной матрицы
!pip install phik

In [ ]:
import phik
from phik.report import plot_correlation_matrix

In [ ]:
# для отлеживания процесса обработки данных (прогресс-бар)
from tqdm import tqdm
tqdm.pandas()

##2. Сбор корпуса

Корпус был собран из двух источников:  

* **Digitale Bibliotheek voor de Nederlandse Letteren (DBNL), publiek domein** - база текстов, размещенная на сайте Электронной библиотеки нидерландской литературы; находится в публичном доступе, содержит тексты разных жанров - романы, рассказы, пьесы, поэтические произведения, эссе, научную литературу; тексты датируются XIII-XIX вв.

* **Corpus Middelnederlands (CM)** -  база текстов, написанных на средненидерландском (Middelnederlands) языке; доступна на сайте Института нидерландского языка (Instituut voor de Nederlandse Taal, INT); содержит рассказы, поэтические произведения, пьесы; тексты датируются XIII-XVI вв.

Из-за ограничений оперативной памяти из баз данных взята часть произведений общим объемом в **855 текстов**.

Корпус собирался в следующем порядке:  
1) С сайта Электронной библиотеки нидерландской литературы загружена находящаяся в открытом доступе **база текстов** (в формате .txt), а также **метаданные** (в формате .csv) для этой базы, в которой содержатся:  
- уникальный номер и название произведения,  
- автор произведения (при наличии);  
- пол автора (0 - мужчина, 1 - женщина);  
- год издания произведения;  
- жанр произведения;  
- библиотека, в которой хранится экземляр произведения, и др.

2) Тексты произведений собраны в таблицу в формате .csv с двумя колонками - уникальный номер произведения и текст произведения.

3) Таблица с метаданными и таблица с текстами произведений **объединены** (merge) по уникальному номеру произведения.

4) Объединенная таблица **дополнена** текстами из базы текстов, написанных на средненидерландском языке, в той части, в которой этих текстов не было на сайте Электронной библиотеки нидерландской литературы; тексты из базы текстов, написанных на средненидерландском языке, были получены путем парсинга .xml файлов с помощью BeautifulSoup.  

5) Из финальной таблицы удалены **дубликаты** (дубликаты в финальной таблице возникли в связи с тем, что в таблице с метаданными сведения о произведениях хранятся в длинном формате: если у произведения несколько авторов или оно относится к нескольким жанрам, то сведения о нем распределены по нескольким строкам).

*Таблица с метаданными*

In [ ]:
df_metadata = pd.read_csv('/content/titels_pd.csv', sep = '|', skiprows=1, on_bad_lines="skip")

In [ ]:
df_metadata.columns

*Таблица с текстами*

In [ ]:
list_texts = []
list_ti_ids = []
count = 0

for f in os.listdir():
  if f.endswith('.txt'):
    with open(f'/content/{f}', 'r', encoding='utf-8') as file_by_one:
      list_texts.append(file_by_one.read())
      list_ti_ids.append(f.split('_01.')[0])
#ti_id - уникальные идентификаторы произведений и одновременно названия файлов, за исключением пометки в конце _01

    count += 1

    print(f'{count}')

In [ ]:
df_id_and_text = pd.DataFrame({
    'ti_id': list_ti_ids,
    'text': list_texts
})

In [ ]:
# df_id_and_text.to_csv('df_id_and_text.csv', index=False)

*Объединенная таблица (метаданные и тексты)*

In [ ]:
df_merged = pd.merge(df_id_and_text, df_metadata, on='ti_id', how='inner')

*Дополнение корпуса*

In [ ]:
files = ['roman_van_limborch.xml', 'van_saladijn.xml',  'vierde_martijn.xml', 'der_leken_spieghel.xml', 'vanden_kerstenen_ghelove.xml', 'getijdenboek_van_geert_grote.xml', 'visioenen.xml',  'brieven.xml', 'mengeldichten_hadewijch.xml', 'alexanders_geesten.xml', 'spiegel_historiael__2.xml', 'spiegel_historiael__1_3_4_maerlant.xml', 'spiegel_historiael__5.xml', 'rinclus.xml', 'der_minnen_loep.xml', 'mellibeus.xml', 'vanden_vier_becoringhen.xml', 'vanden_seven_sloten.xml',  'vanden_twaelf_beghinen.xml', 'vanden_blinckenden_steen.xml', 'van_seven_trappen.xml', 'aiol.xml', 'aubri_de_borgengoen.xml', 'boudewijn_van_seborch_fragm_m.xml', 'roman_van_cassamus__lang_1360.xml', 'des_coninx_summe.xml', 'ferguut.xml', 'floovent.xml', 'gwidekijn_van_sassen.xml',  'huge_van_bordeeus_fragm_m.xml', 'roman_van_lancelot.xml', 'lanceloet_en_het_hert_met_de_witte_voet.xml', 'roman_der_lorreinen_fragm_gi.xml', 'loyhier_en_malaert_fragm_g.xml', 'die_dietsche_lucidarius.xml',  'historie_van_malegijs.xml', 'roman_van_moriaen.xml', 'ons_heren_passie.xml', 'vlaamse_rose_fragm_beu.xml',  'segheliin_van_jerusalem.xml', 'tondalus_visioen_hs_n.xml', 'roman_van_torec.xml', 'truwanten.xml', 'valentijn_en_nameloos_fragm_ge.xml',  'van_smeinscen_lede.xml', 'van_den_vos_reynaerde.xml', 'willem_van_oringen.xml']

In [ ]:
list_texts_to_add = []
list_titles = []

for file_name in files:

    with open(f"D:\\Компьютерная лингвистика\\Машинное обучение\\проект\\корпуса\\CorpusMiddelnederlands_1.0\\CorpusMiddelnederlands_1.0\\all\\{file_name}", "r", encoding="utf-8") as file:
        page_n = file.read()
    soup = BeautifulSoup(page_n, 'lxml-xml')
    text = soup.find('text').text
    title = soup.find('title').text
    file_n = file_name.split('.x')[0]

    list_texts_to_add.append(text)
    list_titles.append(title)

    with open(f"D:\\Компьютерная лингвистика\\Машинное обучение\\проект\\корпуса\\txt_add\\{file_n}.txt", "w", encoding="utf-8") as file_w:
        file_w.write(f"{title}")
        file_w.write(f"{text}")

In [ ]:
dict_add_mid_ned = {'list_titels': list_titles,
                    'text': list_texts_to_add }

In [ ]:
df_add = pd.DataFrame(dict_add_mid_ned)

Поскольку дополнительных текстов было немного (около 50), ti_id (уникальные идентификаторы произведений) были добавлены в таблицу df_add вручную. Это было также необходимо для того, чтобы перепроверить, что добавляемые тексты соотносятся с метаданными о них и не повторяют тексты, которые уже имеются в корпусе.

In [ ]:
# df_add.to_csv('D:\\Компьютерная лингвистика\\Машинное обучение\\проект\\additional_texts.csv', index = False)

In [ ]:
df_merged_2 = pd.merge(df_metadata, df_add, on='ti_id', how='inner')

In [ ]:
df_merged_all = pd.concat([df_merged, df_merged_2], ignore_index=True)

In [ ]:
df_merged_all = df_merged_all.drop_duplicates(subset=['ti_id'])

In [ ]:
# df_merged_all.to_csv('/content/cleaned_result_all_855_SHORTER.csv', index = False)

##3. Исследование данных и определение таргета

###3.1. Проверка данных

Посмотрим, какие есть в получившемся датафрейме колонки, а также проверим, что все 853 собранных текста есть в датафрейме и нет пропущенных данных, которые понадобятся при обучении моделей.

In [ ]:
df_merged_all = pd.read_csv('/content/cleaned_result_all_855_SHORTER_NEW.csv')

In [ ]:
df_merged_all.columns

In [ ]:
df_merged_all.dtypes
#колонки с годом две: в jaar необязательно числовое значение, может быть написано "примерно 1200 г."
#_jaar - числовое значение

In [ ]:
print(f'Самое раннее произведение датируется {df_merged_all['_jaar'].min()} годом.')
print(f'Самое позднее - {df_merged_all['_jaar'].max()} годом.')

In [ ]:
print(f'Группы произведений - {', '.join(df_merged_all['genre'].unique())}.')
print('То есть нон-фикшен, поэзия, литературоведение, проза, литература для детей и подростков, языкознание, пьесы.')

In [ ]:
df_merged_all['genre'].value_counts(ascending = False)

In [ ]:
cmap = plt.get_cmap('Set2')

my_colors = [cmap(i) for i in range(len(df_merged_all))]

df_merged_all['genre'].value_counts().plot(kind='pie',
                                           figsize=(6, 6),
                                           title="Соотношение групп текстов",
                                           colors=my_colors,
                                           wedgeprops={'linewidth': 0},
                                           labels=['поэзия', 'проза', 'нон-фикшен', 'литературоведение', 'пьесы', 'литература для детей\n и подростков', 'языкознание'])
plt.ylabel('')
plt.show()

Можно отметить, что группы произведений обозначены очень специфически (группы неравновесные - род литературы соседствует с конкретными жанрами), при этом среди книг есть сочинения в области **литературоведения и языкознания**.

Книги в области литературоведения и языкознания потенциально могут создавать шум при обучении модели, поскольку в них могут содержаться цитаты из произведений, датируемых другим веком, словарные статьи для языков, отличных от нидерландского.

На данном этапе проекта, однако, было экспериментально решено оставить такие книги в корпусе и посмотреть, будут ли они искажать данные.

Также  можно обратить внимание на то, что в корпусе много **поэтических произведений**. Это необходимо иметь в виду при предобработке текста (например, при удалении лишних переносов строк), а также при обучении модели, если в качестве признаков подаются слова, поскольку поэтические произведения часто специфичны и используют лексику, отличную от обыденной.

In [ ]:
if_dropped = df_merged_all[(df_merged_all['genre'] != 'sec - letterkunde') & (df_merged_all['genre'] != 'sec - taalkunde')]
if_dropped.shape #размер корпуса, если удалить книги по литературоведению и языкознанию (760 текстов)

In [ ]:
df_merged_all.shape #размер корпуса (853 текста)

In [ ]:
df_merged_all['text'].isna().any()

In [ ]:
df_merged_all['_jaar'].isna().any()

In [ ]:
df_merged_all['vrouw'].isna().any()

###3.2. Отбор данных
Отберем колонки, которыен понадобятся для обучения модели:  
* уникальный идентификатор произведения;
* текст произведения;
* год издания;
* пол автора.

In [ ]:
df_selected = df_merged_all[['ti_id', 'text', '_jaar', 'vrouw']].copy()

###3.3. Выбор таргета

Поскольку язык меняется с течением веков, при этом такие изменения происходят постепенно, в качестве таргета были выбраны **периоды (века)**, в которые было издано произведение.

Конкретные временные периоды были выбраны исходя из:

1) Подходов исследователей нидерландского языка к периодизации его истории, см. в том числе:  

* [Dutch. A linguistic history of Holland and Belgium](https://www.dbnl.org/tekst/dona001dutc02_01/dona001dutc02_01_0004.php), Bruce Donaldson;  
* [Middle Dutch – A short introduction](https://www.researchgate.net/publication/291339060_Middle_Dutch_-_A_short_introduction), Matthias Hüning and Ulrike Vogl (Of Reynaert the Fox: Text and Facing Translation of the Middle Dutch Beast Epic Van den vos Reynaerde, André Bouwman and Bart Besamusca);
* [Negatieverschijnselen en woordvolgorde in de geschiedenis van het Nederlands](https://www.dbnl.org/tekst/hors009nega01_01/hors009nega01_01_0001.php), J.M. van der Horst, Marijke J. van der Wal;
* [Spellinggeschiedenis](https://onzetaal.nl/schatkamer/lezen/taal-en-maatschappij/spelling-geschiedenis), Onze Taal.

2) Самостоятельного анализа текстов, написанных в разное время, их сравнения и выявления признаков, характерых для каждой группы таких текстов.

3) Результатов работы модели с разными временными периодами (изначально была опробована более подробная периодизация с разбивкой не по несколько веков, а по одному веку - до 1600, 1600-1700, 1700-1800, 1800-1900; такая разбивка оказалась неэффективной, поскольку тексты из разных периодов были слишком похожи и модель в них путалась).

In [ ]:
for_column_year = [
    (df_selected['_jaar'] < 1500, '1200-1500 гг.'),
    (df_selected['_jaar'] < 1700, '1500-1700 гг.'),
    (df_selected['_jaar'] < 1900, '1700-1900 гг.')
]

In [ ]:
df_selected['eeuw'] = df_selected['_jaar'].case_when(caselist=for_column_year)

In [ ]:
df_selected = df_selected.drop(columns='_jaar')

In [ ]:
df_selected.columns

In [ ]:
print(df_selected[df_selected['eeuw'] == '1200-1500 гг.'].tail())

###3.4. Выявление соотношения классов

В данных присутствует **дисбаланс классов**, который в дальнейшем необходимо будет учитывать при обучении модели.

In [ ]:
df_selected['eeuw'].value_counts()

In [ ]:
df_selected['eeuw'].value_counts().plot(kind='pie', figsize=(6, 6), title="Соотношение классов")
plt.ylabel('')
plt.show()

##4. Формирование признаков (features) для обучения модели

Признаки для дальнейшего обучения модели были отобраны, как и в случае с выбором таргета, с использованием **научных статей** (об особенностях орфографии, лексики и синтаксиса нидерландского языка в разные века), самостоятельного **анализа** текстов и результатов **работы модели** с разным набором признаков.  

Поиск и подсчет признаков в текстах осуществлялся в первую очередь с помощью регулярных выражений, поскольку:  

* обработка (в частности лемматизация, тегирование POS) исторических текстов сопряжена с необходимостью дополнительного лингвистического препроцессинга текстов либо использования специальных предобученных моделей, которые не всегда находятся в открытом доступе;

*  при этом с учетом характера признаков (см. ниже) использование регулярных выражений было наиболее оптимальным способом поиска признаков в текстах и не снижало качество результата работы модели.




###4.1. Конструкция отрицания (ontkenning)

В нидерландском языке, как и в некоторых других языках (английском, немецком, французском), на протяжении его истории менялась конструкция отрицания.

Она прошла путь от единичного отрицания (*ne, en*, до XII в.) к **двусоставному отрицанию** (*en... niet/geen/niemant etc.*) и затем обратно к единичному (*niet/geen/niemand etc.*).

В связи с этой особенностью нидерландского языка было сформулировано предположение, что количество конструкций *en... niet* будет разниться в зависимости от времени издания произведения: в старых произведениях их будет больше, а в более современных - меньше (в современном нидерландском *en... niet* также может встречаться, но в другом значении: *en* теперь означает не отрицание, а союз "и", и в таком значении встречаемость *en... niet* будет ниже).

Функция **clean_text_initial_and_find_negation**:
* предварительно обрабатывает текст (чистит его от лишних знаков);
* делит текст на предложения;
* ищет двусоставное отрицание в предложениях и считает отношение количества предложений с двусоставным отрицанием к общему количеству предложений в тексте.

In [ ]:
def clean_text_initial_and_find_negation(text):
  try:
    clean_text = text.split('onderscheiden van de rest van de tekst door middel van accolades')[1]
  except:
    clean_text = text
  clean_text = re.sub(r'{[=\w\s><\.:-]+}', ' ', clean_text)
  clean_text = re.sub(r'\d', ' ', clean_text)
  clean_text = re.sub(r'[,\\\/""''><;{}\n\t&%$#@*”“‘’()§=:]+', ' ', clean_text)
  clean_text = re.sub(r'\xa0', ' ', clean_text)
  clean_text = re.sub(r'[\[\]]', '', clean_text)
  clean_text = re.sub(r'[\?\!]', '.', clean_text)
  clean_text = re.sub(r'[^\w\s\.]', ' ', clean_text)
  clean_text = re.sub(r'\b[A-Za-z]\b', ' ', clean_text)
  clean_text = re.sub(r'\n+', ' ', clean_text)
  clean_text = re.sub(r' +', ' ', clean_text)
  clean_text = clean_text.lower()

  sentences = clean_text.split('.')

  num_sentences = len(sentences)
  count_negations = 0
  for sentence in sentences:
    negation = re.findall(r'\ben\b \b\w+\b \bniet\b', sentence)
    if len(negation) > 0:
      count_negations +=1
  count_negations_rel = count_negations / num_sentences

  return count_negations_rel

In [ ]:
df_selected['count_negations_rel'] = df_selected['text'].progress_apply(clean_text_initial_and_find_negation)

In [ ]:
print(df_selected.head())

###4.2. Особенности орфографии

####4.2.1. Предобработка текста

Для дальнейшей работы с датафреймом тексты предобрабатываются:
* удаляется "шапка" текстов с их названием и выходными данными;
* исключаются специальные символы, множественные и неразрывные пробелы, цифры;
* слова восстанавливаются до их полного написания путем удаления квадратных скобок (обычно в квадратных скобках дописываются слова, которые были неразборчивы в оригинале, например *in d[en] Haag*).

Функция **clean_text** частично повторяет **clean_text_initial_and_find_negation**, но содержит свои особенности, поскольку в дальнейшем текст не будет делиться на предложения (в частности, точки в функции clean_text удаляются, в отличие от clean_text_initial_and_find_negation, где они используются для разбивки на предложения с помощью split).

In [ ]:
def clean_text(text):
  try:
    clean_text = text.split('onderscheiden van de rest van de tekst door middel van accolades')[1]
  except:
    clean_text = text
  clean_text = re.sub(r'{[=\w\s><\.:-]+}', ' ', clean_text)
  clean_text = re.sub(r'\d', ' ', clean_text)
  clean_text = re.sub(r'\xa0', ' ', clean_text)
  clean_text = re.sub(r'[\[\]]', '', clean_text)
  clean_text = re.sub(r'[^\w\s]', ' ', clean_text)
  clean_text = re.sub(r'\b[A-Za-z]\b', ' ', clean_text)
  clean_text = re.sub(r'\n+', ' ', clean_text)
  clean_text = re.sub(r' +', ' ', clean_text)
  clean_text = clean_text.lower()
  return clean_text

In [ ]:
df_selected_clean = df_selected.copy()

In [ ]:
df_selected_clean['text'] = df_selected_clean['text'].progress_apply(clean_text)

In [ ]:
print(df_selected_clean['text'].iloc[5][204:279])

####4.2.2. Поиск паттернов

Среди особенностей орфографии в качестве паттернов для поиска и подсчета в текстах были выбраны следующие особенности.

| <font size="2.5"> Буквы (сочетание букв) </font> | <font size="2.5"> Пояснение </font> | <font size="2.5"> Пример </font>|
| -------- | -------- | -------- |
| <font size="3"> y </font> | <font size="3"> заменяла в старых текстах дифтонг ij </font> | <font size="3"> myn = mijn </font> |
| <font size="3"> ae </font> | <font size="3"> обозначало долгую a </font> | <font size="3"> jaer = jaar </font> |
| <font size="3"> ck </font> | <font size="3"> с течением времени заменено на k </font> | <font size="3"> oock = ook </font> |
| <font size="3"> gh </font> | <font size="3"> стало обозначаться g </font> | <font size="3"> ghenoech = genoeg </font> |
| <font size="3"> sch </font> | <font size="3"> во многих случаях заменено на s </font> | <font size="3"> mensch = mens </font> |
| <font size="3"> z </font> | <font size="3"> z почти не использовалась </font> | <font size="3"> seer = zeer </font> |
| <font size="3"> nt </font> | <font size="3"> t не озвончалось </font> | <font size="3"> hant = hand </font> |
| <font size="3"> kw </font> | <font size="3"> использовалось сочетание qu </font> | <font size="3"> quam = kwam </font> |
| <font size="3"> aa, ee, oo </font> | <font size="3"> заменены в открытом слоге на a, e, o </font> | <font size="3"> heeten = heten </font> |

Функция **count_patterns** принимает на вход текст и паттерн, считает количество паттернов в тексте и выдает отношение количества паттернов в тексте к общему количеству символов (без пробелов) в тексте.

In [ ]:
def count_patterns(text_to_pr, pattern):
  count_absolute = len(re.findall(fr'{pattern}', text_to_pr))
  symbols = re.sub(' ', '', text_to_pr)
  count_all = len(symbols)
  count_relative = count_absolute / count_all
  return count_relative

In [ ]:
df_selected_patterns = df_selected_clean.copy()

In [ ]:
patterns = ['y', 'ae', 'ck', 'gh', 'sch', 'z', 'nt ', 'kw', 'aa', 'ee', 'oo']

In [ ]:
for patt in tqdm(patterns):
  df_selected_patterns[patt] = df_selected_patterns['text'].apply(lambda x: count_patterns(text_to_pr = x, pattern = patt))

Можно посмотреть предварительно распределение паттернов по классам, для наглядности. В дальнейшем все паттерны будут подсчитаны в рамках корреляционной матрицы.

In [ ]:
sns.stripplot(data=df_selected_patterns, x='eeuw', y='z', jitter=True, size = 3)

plt.show()

In [ ]:
sns.stripplot(data=df_selected_patterns, x='eeuw', y='nt ', jitter=True, size = 3)

plt.show()

In [ ]:
print(df_selected_patterns.head())

###4.3. Особенности лексики и грамматики

####4.3.1. Разбивка текстов на токены

Для поиска лексических и грамматических признаков в текстах тексты были разбиты на наивные токены.  

Функция **naive_tokens** делит текст на токены по пробелу и отбирает среди них в окончательный список токены длиной более 1 символа.

In [ ]:
def naive_tokens(text):
  tokens = text.split()
  clean_tokens = []
  for i in tokens:
    if len(i) > 1:
      clean_tokens.append(i)
  return clean_tokens

In [ ]:
df_selected_patterns_tokens = df_selected_patterns.copy()

In [ ]:
df_selected_patterns_tokens['tokens'] = df_selected_patterns_tokens['text'].progress_apply(naive_tokens)

In [ ]:
print(df_selected_patterns_tokens.head())

####4.3.2. Подсчет TTR (type-token ratio)

Поскольку раньше в нидерландском языке было больше падежных форм и форм глаголов, было сделано предположение, что **TTR** при подсчете словоформ на старых текстах может быть выше, чем на более современных.  

В то же время TTR может оказаться непоказательным с учетом повышения со временем лексического разнообразия языка (в том числе за счет появления новых слов, заимствований).

Функция **calculate_ttr** подсчитывает TTR текста, предварительно разбив его на батчи размером 250 и менее токенов:  
* если размер батча менее 250 и это единственный батч в тексте, то его TTR признается NaN и в дальнейшем для такого текста TTR будет определен путем усреднения значений по классу;
* если размер батча менее 250 и это не единственный батч в тексте, то его TTR признается равным нулю и не включается подсчет общего TTR по тексту;
* если размер батча равен 250, то TTR считается стандартно (количество уникальных токенов делится на общее количество токенов).

Такой подход к подсчету TTR используется для того, чтобы размер текстов и батчей не влиял на объективный подсчет TTR (потому что на слишком длинных текстах TTR падает, а на слишком коротких батчах - может завышаться).

In [ ]:
def calculate_ttr(list_tokens):
  chunk_size = 250
  chunks = list(batched(list_tokens, chunk_size))

  ttr_per_chunk = []
  for chunk in chunks:
    num_tokens = len(chunk)
    num_types = len(set(chunk))
    if len(chunk) < 250:
      if len(chunks) == 1:
        ttr_1 = float('NaN')
      else:
        ttr_1 = 0
    else:
      ttr_1 = num_types / num_tokens
    ttr_per_chunk.append(ttr_1)

  if np.isnan(ttr_per_chunk[0]):
    ttr = float('NaN')
  else:
    if ttr_per_chunk[-1] == 0:
      ttr = sum(ttr_per_chunk) / (len(ttr_per_chunk)-1)
    else:
      ttr = sum(ttr_per_chunk) / len(ttr_per_chunk)

  return ttr

In [ ]:
df_selected_patterns_tokens['ttr'] = df_selected_patterns_tokens['tokens'].progress_apply(calculate_ttr)

In [ ]:
df_selected_patterns_tokens['ttr'] = df_selected_patterns_tokens['ttr'].fillna(df_selected_patterns_tokens.groupby('eeuw')['ttr'].transform('mean'))

In [ ]:
print(df_selected_patterns_tokens.head())

####4.3.3. Слова-индикаторы

Некоторые **слова** можно использовать как индикаторы для отнесения текста к определенному веку.  

В качестве примера взято слово *ende* : раньше в текстах оно использовалось в роли союза "и". В современном нидерландском языке его постепенно вытеснило *en*.  

Функция **count_words** считает относительную встречаемость слова среди всех токенов текста.

In [ ]:
def count_words(list_tokens, to_find):
  count_words = 0
  for token in list_tokens:
    if token == to_find:
      count_words += 1
  count_words_rel = count_words / len(list_tokens)
  return count_words_rel

In [ ]:
df_selected_patterns_tokens['ende'] = df_selected_patterns_tokens['tokens'].progress_apply(lambda x: count_words(list_tokens = x, to_find = 'ende'))

In [ ]:
print(df_selected_patterns_tokens.head())

####4.3.4. Формы слов

Помимо непосредственно слов, определенные **словоформы** также могут указывать на век, в котором было издано произведение.  

Например, раньше в косвенных падежах местоимение *ik* принимало форму не *mij* (*me*), а *mi* (в винительном и дательном падеже); "ты" в косвенном падеже было *di*.

Также много словоформ оканчивалось на *-e*, в отличие от современного нидерландского языка. В частности, одним из способов образования множественного числа было прибавление к слову окончания *-e* (*dach* - *dage*). В настоящее время множественное число формируется путем прибавления *-en* либо *-s*.

Более распространены были падежные формы артиклей - *der, des, den*. В современном языке они остаются в основном в устойчивых конструкциях (как в названии Гааги - den Haag).

In [ ]:
def count_bag_of_words(tokens, regex_words):
  count_all_tokens = len(tokens)
  joined_text = ' '.join(tokens)
  count_words_absolute = len(re.findall(fr'{regex_words}', joined_text))
  count_words_relative = count_words_absolute / count_all_tokens
  return count_words_relative

In [ ]:
additional_patterns = {'mi, di': r'\bmi\b|\bdi\b',
                       '-e': r'\b\w+e\b',
                       'des, der, den': r'\bdes\b|\bder\b|\bden\b'
                       }

In [ ]:
for key, value in tqdm(additional_patterns.items()):
  df_selected_patterns_tokens[key] = df_selected_patterns_tokens['tokens'].apply(lambda x: count_bag_of_words(tokens = x, regex_words = value))

In [ ]:
sns.stripplot(data=df_selected_patterns_tokens, x='eeuw', y='-e', jitter=True, size = 3)

plt.show()

In [ ]:
# to_save = df_selected_patterns_tokens.drop(columns=['tokens', 'text'])

In [ ]:
# to_save.to_excel('res_without_texts_2.xlsx', index = False)

##5. Подготовка данных для обучения модели

###5.1. Отбор признаков для обучения модели

Для обучения модели важно отобрать признаки, которые характеризуются следующим:
* с одной стороны, признаки должны действительно помогать определять таргет, **коррелировать** с ним (применительно к данному проекту - век, в котором издано произведение), то есть не должны быть "шумом", мешающим модели сделать правильное предсказание;
* с другой строны, признаки между собой **не должны быть мультиколлинеарны**, иными словами, не должны сильно коррелировать друг с другом.

Для отбора признаков, отвечающих вышеназванным критериям, была построена корреляционная матрица.

По итогам ее анализа было принято решение отказаться от такого признака, как пол автора произведения, поскольку он очень мало коррелирует с таргетом (0,033).

Остальные признаки решено оставить, при этом признаки, сильно коррелирующие с таргетом, оставлены для линейных моделей, для нелинейных моделей - все признаки.

Признаков мультиколлинеарности между признаками (значения выше 0,8%) не выявлены.

In [ ]:
df_selected_patterns_tokens.columns

In [ ]:
df_for_phik = df_selected_patterns_tokens.drop(columns = ['ti_id', 'text', 'tokens'])

In [ ]:
phik_matrix = df_for_phik.phik_matrix()

plt.figure(figsize=(10,8))
sns.heatmap(phik_matrix, annot=True, cmap='coolwarm')
plt.title('Phik Correlation Matrix')
plt.show()

In [ ]:
linear = ['eeuw', 'y', 'ae', 'ck', 'gh', 'sch', 'z', 'kw', 'nt ', 'ee', 'oo', 'mi, di', '-e', 'aa', 'ende']

In [ ]:
linear_features = ['y', 'ae', 'ck', 'gh', 'sch', 'z', 'kw', 'nt ', 'ee', 'oo', 'mi, di', '-e', 'aa', 'ende']

In [ ]:
df_to_process_linear = df_selected_patterns_tokens[linear]

In [ ]:
nonlinear = ['eeuw', 'y', 'ae', 'ck', 'gh', 'sch', 'z', 'kw', 'nt ', 'ee', 'oo', 'mi, di', '-e', 'aa', 'ende', 'des, der, den', 'ttr', 'count_negations_rel']

In [ ]:
nonlinear_features = ['y', 'ae', 'ck', 'gh', 'sch', 'z', 'kw', 'nt ', 'ee', 'oo', 'mi, di', '-e', 'aa', 'ende', 'des, der, den', 'ttr', 'count_negations_rel']

In [ ]:
df_to_process_nonlinear = df_selected_patterns_tokens[nonlinear]

###5.2. Деление данных на тренировочную часть (train) и тестовую часть (test)
Данные поделены на train и test с размером тест-части в 20% и одинаковым распределением классов в train и test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_to_process_linear[linear_features], df_to_process_linear['eeuw'], test_size=0.2, random_state=42, stratify=df_to_process_linear['eeuw'])

In [ ]:
X_train.shape

In [ ]:
X_train.columns

In [ ]:
X_test.shape

In [ ]:
X_train_nl, X_test_nl, y_train_nl, y_test_nl = train_test_split(df_to_process_nonlinear[nonlinear_features], df_to_process_nonlinear['eeuw'], test_size=0.2, random_state=42, stratify=df_to_process_nonlinear['eeuw'])

In [ ]:
X_train_nl.shape

In [ ]:
X_test_nl.shape

###5.3. Масштабирование количественных признаков
Количественные признаки масштабированы с помощью MinMaxScaler(), а не StandardScaler(), чтобы они оставались положительными и могли быть переданы в дальнейшем в том числе в модель Мультиноминального Байеса, который работает только с положительными величинами.  

Обучение скейлера проводилось на train-части данных.


In [ ]:
def to_scale(features_train, features_test):
  scaler = MinMaxScaler()
  X_train_scaled = scaler.fit_transform(features_train)
  X_test_scaled = scaler.transform(features_test)
  return X_train_scaled, X_test_scaled

In [ ]:
X_lin = to_scale(X_train, X_test)
X_nonlin = to_scale(X_train_nl, X_test_nl)

In [ ]:
lin[0].shape

In [ ]:
lin[1].shape

In [ ]:
# scaler = MinMaxScaler()

In [ ]:
# X_train_scaled = scaler.fit_transform(X_train)

In [ ]:
# X_test_scaled = scaler.transform(X_test)

In [ ]:
# X_train_scaled[0]

In [ ]:
# X_train_scaled.shape

In [ ]:
# X_test_scaled[0]

In [ ]:
# X_test_scaled.shape

###5.4. Кодирование таргета
Поскольку таргет в данном проекте - не количественная величина, он был закодирован в числа с помощью LabelEncoder().  

Обучение энкодера проводилось на train-части данных.

In [ ]:
def to_encode(target_train, target_test):
  le = LabelEncoder()
  y_train_encoded = le.fit_transform(target_train)
  y_test_encoded = le.transform(target_test)
  return y_train_encoded, y_test_encoded, le.classes_

In [ ]:
y_lin = to_encode(y_train, y_test)  #в данном случае y_lin и y_nonlin будут одинаковые, но для чистоты разделения lin/nonlin оставлю разбивку
y_nonlin = to_encode(y_train_nl, y_test_nl)

In [ ]:
y_lin[2]

In [ ]:
# le = LabelEncoder()

In [ ]:
# le.fit(y_train)

In [ ]:
# le.classes_

In [ ]:
# y_train_encoded = le.transform(y_train)
# y_test_encoded = le.transform(y_test)

##6. Обучение моделей и получение метрик
###6.1. Модели

Для классификации текстов по векам были опробованы несколько классических моделей:
1. Логистическая регрессия с балансом классов;
2. Мультиноминальный Байес с подсчетом баланса классов с помощью sample_weight;
3. Мультиноминальный Байес с приоритетом класса (в пользу наиболее сложно определяемых классов);
4. K-ближайших соседей (со стандартными аргументами для работы с текстами, в частности косинусной метрикой);
5. Дерево решений.

В рамках функции **model_and_metrics** модель обучается, делает предсказание, и на выходе функция выдает метрики (accuracy, f1-score и полный classification report).

In [ ]:
s_weight = compute_sample_weight('balanced', y_lin[0]) #lin и nonlin одинаковые

In [ ]:
def model_and_metrics(model, x_tr, x_t, y_tr, y_t):

  model_to_use = model
  if model == MultinomialNB():
    model_to_use.fit(x_tr, y_tr, sample_weight = s_weight) #обучаем модель, если она просто мультиномиальный Байес, без priors
  else:
    model_to_use.fit(x_tr, y_tr) #обучаем модель, если она не MultinomialNB()

  y_pred = model_to_use.predict(x_t)

  accuracy = accuracy_score(y_t, y_pred) #сравниваем предсказание с настоящим значением
  f1 = f1_score(y_t, y_pred, average='weighted')

  cl = classification_report(y_t, y_pred, target_names=y_lin[2]) #lin и nonlin одинаковые

  cl_dict = classification_report(y_t, y_pred, output_dict=True, target_names=y_lin[2])

  return accuracy, f1, cl, cl_dict

In [ ]:
all_models_linear = {'Логистическая регрессия (регуляризатор - l2, Ридж, с балансом классов)': LogisticRegression(random_state=42, class_weight='balanced', max_iter=700),
              'Мультиноминальный Байес с sample_weight': MultinomialNB(),
              'Мультиноминальный Байес с приоритетом класса': MultinomialNB(class_prior=np.array([0.3, 0.4, 0.3]))}

In [ ]:
all_models_nonlinear = {'K-ближайших соседей': KNeighborsClassifier(n_neighbors=7, metric = 'cosine', weights = 'distance'),
                      'Decision Tree': DecisionTreeClassifier(random_state = 42, max_depth=3)}

In [ ]:
all_res_linear = {}
for key, value in tqdm(all_models_linear.items()):
  res = model_and_metrics(value, X_lin[0], X_lin[1], y_lin[0], y_lin[1])
  all_res_linear.update({key: [res[0], res[1], res[2], res[3]]})

In [ ]:
for_df_metrics_lin = {k: v[:-2] for k, v in all_res_linear.items()}
for_cl_all_lin = {k: v[-2:] for k, v in all_res_linear.items()}

In [ ]:
all_res_nonlinear = {}
for key, value in tqdm(all_models_nonlinear.items()):
  res = model_and_metrics(value, X_nonlin[0], X_nonlin[1], y_nonlin[0], y_nonlin[1])
  all_res_nonlinear.update({key: [res[0], res[1], res[2], res[3]]})

In [ ]:
for_df_metrics_nonlin = {k: v[:-2] for k, v in all_res_nonlinear.items()}
for_cl_all_nonlin = {k: v[-2:] for k, v in all_res_nonlinear.items()}

###6.2. Метрики
Все модели показали себя в разной степени хорошо, с показателями в основном более 80%.  
Тем не менее, модели стоит опробовать на большем объеме данных, чтобы избежать ошибки переобучения.

####6.2.1. Accuracy и f1-score

*Линейные модели*

In [ ]:
df_metrics_lin = pd.DataFrame.from_dict(for_df_metrics_lin).T.rename(columns={0: "Accuracy", 1: "F1-score"})

In [ ]:
# df_metrics_lin.to_excel('metrics_lin.xlsx')

In [ ]:
df_metrics_lin

*Нелинейные модели*

In [ ]:
df_metrics_nonlin = pd.DataFrame.from_dict(for_df_metrics_nonlin).T.rename(columns={0: "Accuracy", 1: "F1-score"})

In [ ]:
df_metrics_nonlin.to_excel('metrics_nonlin.xlsx')

In [ ]:
df_metrics_nonlin

####6.2.2. Classification report

*Линейные модели*

In [ ]:
for_cl_lin = {k: v[:-1] for k, v in for_cl_all_lin.items()}
for_cl_df_lin = {k: v[-1:] for k, v in for_cl_all_lin.items()}

In [ ]:
for key, value in for_cl_df_lin.items():
  df = pd.DataFrame(value[0]).T
  df.to_excel(f'{key}.xlsx')

In [ ]:
for key, value in for_cl_lin.items():
  print(key)
  print()
  print(value[0])
  print()
  print('--------------------------------------------------\n')

*Нелинейные модели*

In [ ]:
for_cl_nonlin = {k: v[:-1] for k, v in for_cl_all_nonlin.items()}
for_cl_df_nonlin = {k: v[-1:] for k, v in for_cl_all_nonlin.items()}

In [ ]:
for key, value in for_cl_df_nonlin.items():
  df = pd.DataFrame(value[0]).T
  df.to_excel(f'{key}.xlsx')

In [ ]:
for key, value in for_cl_nonlin.items():
  print(key)
  print()
  print(value[0])
  print()
  print('--------------------------------------------------\n')

##7. Ансамбли
Можно попробовать несколько улучшить отдельные метрики и повысить устойчивость моделей с помощью ансамблей.

###7.1. Стекинг
В качестве базовых моделей были выбраны Мультиноминальный Байес с приоритетом классов и К-ближайших соседей, в качестве мета-модели, которая принимает окончательное решение, - логистическая регрессия.

Показатели в целом немного повысились.

In [ ]:
base_models = [
    ('NB', MultinomialNB(class_prior=np.array([0.3, 0.4, 0.3]))),
    ('knn', KNeighborsClassifier(n_neighbors=7, weights = 'distance', metric = 'cosine'))
]

meta_model = LogisticRegression(random_state=42, class_weight='balanced', max_iter=700)

stacking_models = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=2,
)

In [ ]:
stacking_models.fit(X_nonlin[0], y_nonlin[0]) #для обучения возьмем все признаки, то есть те, которые выше использовались для нелинейных моделей

In [ ]:
y_pred_stacking = stacking_models.predict(X_nonlin[1])

In [ ]:
print(classification_report(y_nonlin[1], y_pred_stacking, target_names=y_nonlin[2]))

In [ ]:
class_rep_2 = classification_report(y_nonlin[1], y_pred_stacking, target_names=y_nonlin[2], output_dict=True)
df = pd.DataFrame(class_rep_2).T
# df.to_excel('stacking.xlsx')

###7.2. Бэггинг
Для бэггинга был выбран Мультиномиальный Байес с подсчетом баланса классов с помощью sample_weight. Количество моделей, которые обучаются на разных выборках, - 3.

Метрики в целом стабильны для класса "1700-1900 гг." и несколько  перераспределились по классам "1200-1500 гг." и "1500-1700 гг.".

In [ ]:
bagging = BaggingClassifier(
    estimator= MultinomialNB(),
    n_estimators=3,
    max_samples=1.0,      # размер подвыборки
    bootstrap=True,       # с возвращением
    bootstrap_features=False,
    n_jobs=-1
)

In [ ]:
bagging.fit(X_lin[0], y_lin[0], sample_weight=s_weight)
y_pred_bagging = bagging.predict(X_lin[1])

In [ ]:
print(classification_report(y_lin[1], y_pred_bagging, target_names=y_lin[2]))

In [ ]:
class_rep_3 = classification_report(y_lin[1], y_pred_bagging, target_names=y_lin[2], output_dict = True)
df = pd.DataFrame(class_rep_3).T
# df.to_excel('bagging.xlsx')

###7.3. Бустинг

####7.3.1. Адабустинг
Для адабустинга был выбран Мультиномиальный Байес с подсчетом баланса классов с помощью sample_weight. В рамках адабустинга предполагается, что каждая последующая модель нацелена на исправление ошибок предыдущей модели путем перераспределения весов.

Адабустинг на Мультиномиальном Байесе показал сравнимую с бэггингом результативность.

In [ ]:
adaboost_model = AdaBoostClassifier(
    estimator=MultinomialNB(),
    n_estimators=150,
    learning_rate=0.001
)

In [ ]:
adaboost_model.fit(X_lin[0], y_lin[0], sample_weight=s_weight)
y_pred_adaboost_model = adaboost_model.predict(X_lin[1])

In [ ]:
print(classification_report(y_lin[1], y_pred_adaboost_model, target_names=y_lin[2]))

In [ ]:
class_rep_4 = classification_report(y_lin[1], y_pred_adaboost_model, target_names=y_lin[2], output_dict = True)
df = pd.DataFrame(class_rep_4).T
# df.to_excel('adaboosting.xlsx')

####7.3.2. Градиентный бустинг

В основе градиентного бустинга - деревья решений. Суть же ансамбля та же, что и в адабустинге - корректировка модели после каждого предсказания.

Градиентный бустинг показал небольшое улучшение результата работы дерева решений.

In [ ]:
boosting = GradientBoostingClassifier(n_estimators = 100, learning_rate = 0.01, max_depth = 7)

In [ ]:
boosting.fit(X_nonlin[0], y_nonlin[0])
y_pred_boosting = boosting.predict(X_nonlin[1])

In [ ]:
print(classification_report(y_nonlin[1], y_pred_boosting, target_names=y_nonlin[2]))

In [ ]:
class_rep_5 = classification_report(y_nonlin[1], y_pred_boosting, target_names=y_nonlin[2], output_dict = True)
df = pd.DataFrame(class_rep_5).T
# df.to_excel('boosting.xlsx')